In [ ]:
from __future__ import annotations
import os
import re
import io
import json
import csv
import itertools
import mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator, Union
from datetime import datetime
from dataclasses import dataclass, field, asdict
import argparse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from collections import Counter
import time

# ============================================================================
# Конфигурации и константы
# ============================================================================

INCLUDE_EXTS = {'doc', 'docx', 'gif', 'html', 'ipynb', 'jpeg', 'jpg', 'pdf', 
                'php', 'png', 'rtf', 'xls', 'xlsx', 'txt', 'csv', 'json', 'tif', 'tiff'}

PDN_CATEGORIES = ['обычные', 'государственные', 'платёжные', 'биометрические', 'специальные']

# ============================================================================
# Spark инициализация
# ============================================================================

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import udf, col, size, when, lit
    from pyspark.sql.types import (
        StructType, StructField, StringType, IntegerType, 
        MapType, BooleanType, LongType, ArrayType
    )
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print("PySpark не установлен.")

# ============================================================================
# YOLO инициализация
# ============================================================================

try:
    from ultralytics import YOLO
    import cv2
    import numpy as np
    from PIL import Image
    YOLO_AVAILABLE = True
    _yolo_model = None
    
    def get_yolo_model():
        global _yolo_model
        if _yolo_model is None:
            try:
                _yolo_model = YOLO('yolov8n-face.pt')
            except:
                try:
                    _yolo_model = YOLO('yolov8n.pt')
                except:
                    print("⚠️ Не удалось загрузить модель YOLO")
                    return None
        return _yolo_model
    
except ImportError:
    YOLO_AVAILABLE = False
    print("Ultralytics YOLO не установлен.")
    
    def get_yolo_model():
        return None

# ============================================================================
# Вспомогательные функции
# ============================================================================

def detect_encoding(raw_bytes: bytes) -> str:
    try:
        import chardet
        res = chardet.detect(raw_bytes)
        return res.get('encoding') or 'utf-8'
    except:
        return 'utf-8'

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if not (13 <= len(digits) <= 19):
        return False
    s = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        s += d
    return s % 10 == 0

def snils_valid(snils: str) -> bool:
    nums = re.sub(r'\D', '', snils)
    if len(nums) != 11:
        return False
    base = [int(x) for x in nums[:9]]
    check = int(nums[9:])
    s = sum((9 - i) * d for i, d in enumerate(base))
    if s < 100:
        c = s
    elif s in (100, 101):
        c = 0
    else:
        c = s % 101
        if c == 100:
            c = 0
    return c == check

def inn_valid(inn: str) -> bool:
    nums = re.sub(r'\D', '', inn)
    if len(nums) == 10:
        w = [2, 4, 10, 3, 5, 9, 4, 6, 8]
        c = sum(int(nums[i]) * w[i] for i in range(9)) % 11 % 10
        return c == int(nums[9])
    elif len(nums) == 12:
        w1 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        w2 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        c1 = sum(int(nums[i]) * w1[i] for i in range(11)) % 11 % 10
        c2 = sum(int(nums[i]) * w2[i] for i in range(11)) % 11 % 10
        return c1 == int(nums[10]) and c2 == int(nums[11])
    return False

def has_context(text: str, idx: int, window: int, *keywords: str) -> bool:
    start = max(0, idx - window)
    end = min(len(text), idx + window)
    chunk = text[start:end].lower()
    return any(k.lower() in chunk for k in keywords)

# ============================================================================
# Детекция лиц с помощью YOLO
# ============================================================================

def detect_faces_in_image(image_bytes: bytes) -> Tuple[int, List[Dict]]:
    """
    Детектирует лица на изображении с помощью YOLO
    
    Returns:
        Tuple[int, List[Dict]]: (количество лиц, список с координатами)
    """
    if not YOLO_AVAILABLE:
        return 0, []
    
    model = get_yolo_model()
    if model is None:
        return 0, []
    
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        if img is None:
            return 0, []
        
        results = model(img, classes=[0], conf=0.25)
        
        faces = []
        face_count = 0
        
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                face_count = len(boxes)
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = float(box.conf[0])
                    faces.append({
                        'bbox': [int(x1), int(y1), int(x2), int(y2)],
                        'confidence': conf
                    })
        
        return face_count, faces
        
    except Exception as e:
        print(f"Ошибка детекции лиц: {e}")
        return 0, []

# ============================================================================
# Извлечение текста из файлов (с улучшенной обработкой изображений)
# ============================================================================

def extract_text_from_bytes(content: bytes, ext: str, detect_faces: bool = True) -> Tuple[str, int]:
    face_count = 0
    
    try:
        if ext in {'txt', 'csv', 'json', 'log', 'md', 'py', 'js', 'html', 'xml', 'php', 'rtf'}:
            for enc in ['utf-8', 'cp1251', 'latin-1']:
                try:
                    return content.decode(enc, errors='ignore'), 0
                except:
                    continue
            return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'pdf':
            try:
                import PyPDF2
                import io
                reader = PyPDF2.PdfReader(io.BytesIO(content))
                text = ''
                for page in reader.pages:
                    try:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + '\n'
                    except:
                        pass
                if text.strip():
                    return text, 0
            except:
                pass
            
            try:
                from pdfminer.high_level import extract_text as pdfminer_extract
                import io
                text = pdfminer_extract(io.BytesIO(content)) or ''
                if text.strip():
                    return text, 0
            except:
                pass
            
            return '', 0
        
        elif ext in {'xls', 'xlsx', 'xlsm'}:
            try:
                import pandas as pd
                import io
                df = pd.read_excel(io.BytesIO(content), header=None, dtype=str)
                return ' '.join(df.stack().astype(str).tolist()), 0
            except:
                return '', 0
        
        elif ext == 'docx':
            try:
                import docx
                import io
                doc = docx.Document(io.BytesIO(content))
                parts = []
                for p in doc.paragraphs:
                    if p.text:
                        parts.append(p.text)
                for tbl in doc.tables:
                    for row in tbl.rows:
                        row_text = ' \t '.join(cell.text for cell in row.cells if cell.text)
                        if row_text:
                            parts.append(row_text)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext == 'doc':
            try:
                text = content.decode('latin-1', errors='ignore')
                text = re.sub(r'[^\x20-\x7E\u0400-\u04FF]+', ' ', text)
                return re.sub(r'\s+', ' ', text), 0
            except:
                return '', 0
        
        elif ext == 'html':
            try:
                txt = content.decode('utf-8', errors='ignore')
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(txt, 'html.parser')
                return soup.get_text(' '), 0
            except:
                return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'ipynb':
            try:
                data = json.loads(content.decode('utf-8', errors='ignore'))
                parts = []
                for cell in data.get('cells', []):
                    src = cell.get('source', [])
                    if isinstance(src, list):
                        parts.append(''.join(src))
                    elif isinstance(src, str):
                        parts.append(src)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext in {'jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff', 'bmp'}:
            text = ''
            # OCR для текста
            try:
                import pytesseract
                from PIL import Image
                import io
                img = Image.open(io.BytesIO(content))
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                if img.width * img.height > 4000000:
                    img.thumbnail((2000, 2000))
                text = pytesseract.image_to_string(img, lang='rus+eng')
            except:
                pass
            
            if detect_faces:
                face_count, _ = detect_faces_in_image(content)
            
            return text, face_count
        
        else:
            return content.decode('utf-8', errors='ignore'), 0
            
    except Exception as e:
        return '', 0

# ============================================================================
# Детекторы ПДн
# ============================================================================

EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
DOB_RE = re.compile(r"\b(\d{2}[./]\d{2}[./]\d{4})\b")
INDEX_RE = re.compile(r"\b\d{6}\b")
SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b|\b\d{11}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)

BIOMETRIC_KEYS = [
    'биометр', 'отпечат', 'радуж', 'ирис', 'лицев', 'селфи', 'faceid', 
    'fingerprint', 'iris', 'voiceprint', 'голосов', 'геометрия лица'
]

SPECIAL_KEYS = [
    'диагноз', 'анамнез', 'инвалид', 'здоровь', 'медицин', 'психиатр', 
    'вич', 'религ', 'вероисповед', 'политическ', 'партия', 'интим', 'сексуаль'
]

def count_occurrences(pattern: re.Pattern, text: str) -> int:
    return len(list(pattern.finditer(text)))

def detect_categories(text: str, face_count: int = 0) -> Dict[str, int]:
    t = text if isinstance(text, str) else ''
    low = t.lower()
    cats = {
        'обычные': 0,
        'государственные': 0,
        'платёжные': 0,
        'биометрические': 0,
        'специальные': 0
    }
    
    if face_count > 0:
        cats['биометрические'] += face_count
    
    if not t:
        return cats
    
    cats['обычные'] += count_occurrences(EMAIL_RE, t)
    cats['обычные'] += count_occurrences(PHONE_RE, t)
    cats['обычные'] += min(5, count_occurrences(FIO_RE, t))
    
    for m in DOB_RE.finditer(t):
        if has_context(low, m.start(), 40, 'дата рождения', 'родил', 'д.р.', 'birth'):
            cats['обычные'] += 1
    
    for m in INDEX_RE.finditer(t):
        if has_context(low, m.start(), 40, 'ул', 'улица', 'просп', 'пер', 'дом', 'квартира', 'город', 'г.', 'адрес'):
            cats['обычные'] += 1
    
    for m in SNILS_RE.finditer(t):
        if snils_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN10_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN12_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in PASSPORT_RE.finditer(t):
        if has_context(low, m.start(), 50, 'паспорт', 'серия', 'номер', 'код подразделения'):
            cats['государственные'] += 1
    
    for m in CARD_RE.finditer(t):
        raw = m.group(0)
        digits = re.sub(r'\D', '', raw)
        if 13 <= len(digits) <= 19 and luhn_check(raw):
            if has_context(low, m.start(), 40, 'visa', 'mastercard', 'карта', 'cvv', 'cvc', 'номер карты'):
                cats['платёжные'] += 1
    
    for m in RS_RE.finditer(t):
        cats['платёжные'] += 1
    
    for m in BIK_RE.finditer(t):
        cats['платёжные'] += 1
    
    if CVV_RE.search(t):
        cats['платёжные'] += 1
    
    if any(k in low for k in BIOMETRIC_KEYS):
        cats['биометрические'] += 1
    
    if any(k in low for k in SPECIAL_KEYS):
        cats['специальные'] += 1
    
    return cats

def estimate_uz(cats: Dict[str, int]) -> str:
    total = sum(cats.values())
    distinct = sum(1 for v in cats.values() if v > 0)
    has_special = cats['специальные'] > 0
    has_bio = cats['биометрические'] > 0
    has_pay = cats['платёжные'] > 0
    has_gov = cats['государственные'] > 0
    has_common = cats['обычные'] > 0
    
    if has_special or has_bio:
        return 'УЗ-1' if (total >= 5 or distinct >= 2) else 'УЗ-2'
    if has_pay or has_gov:
        return 'УЗ-2' if (total >= 5 or distinct >= 2) else 'УЗ-3'
    if has_common:
        return 'УЗ-3' if (total >= 5 or distinct >= 2) else 'УЗ-4'
    return 'нет ПДн'

# ============================================================================
# Анализ файла
# ============================================================================

def analyze_file_content(file_path: str, content: bytes, detect_faces: bool = True) -> Dict[str, Any]:
    path = Path(file_path)
    file_stat = path.stat()
    
    result = {
        'path': file_path,
        'name': path.name,
        'size': file_stat.st_size,
        'modified_time': datetime.fromtimestamp(file_stat.st_mtime).strftime('%b %d %H:%M'),
        'success': False,
        'ext': '',
        'text_length': 0,
        'categories': {},
        'total_hits': 0,
        'uz': 'нет ПДн',
        'face_count': 0,
        'error': None
    }
    
    if content is None:
        result['error'] = 'Пустое содержимое'
        return result
    
    ext = path.suffix.lower().lstrip('.')
    result['ext'] = ext
    
    if ext not in INCLUDE_EXTS:
        result['error'] = f'Неподдерживаемый формат: {ext}'
        return result
    
    try:
        text, face_count = extract_text_from_bytes(content, ext, detect_faces)
        result['text_length'] = len(text)
        result['face_count'] = face_count
        
        cats = detect_categories(text, face_count)
        result['categories'] = {k: v for k, v in cats.items() if v > 0}
        result['total_hits'] = sum(cats.values())
        result['uz'] = estimate_uz(cats)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)[:200]
    
    return result

# ============================================================================
# Класс сканера с поддержкой Spark
# ============================================================================

class PDNScanner:
    def __init__(self, max_workers: int = 4, use_spark: bool = True, detect_faces: bool = True):
        self.max_workers = max_workers
        self.use_spark = use_spark and SPARK_AVAILABLE
        self.detect_faces = detect_faces and YOLO_AVAILABLE
        self.results = []
        
        if self.use_spark:
            self.spark = SparkSession.builder \
                .appName("PDNScanner") \
                .config("spark.sql.adaptive.enabled", "true") \
                .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
                .getOrCreate()
            print(f"Spark инициализирован: {self.spark.version}")
        
        if self.detect_faces:
            print("YOLO инициализирован для детекции лиц")
    
    def scan_directory(self, directory_path: str, recursive: bool = True, sample_size: Optional[int] = None) -> pd.DataFrame:
        dir_path = Path(directory_path)
        if not dir_path.exists():
            raise FileNotFoundError(f"Директория не найдена: {directory_path}")
        
        print(f"Сбор файлов из: {dir_path.absolute()}")
        
        files = []
        if recursive:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.rglob(pattern))
        else:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.glob(pattern))
        
        files = list(set(files))
        file_paths = [str(f.absolute()) for f in files]
        
        print(f"Найдено файлов: {len(file_paths)}")
        
        if sample_size and sample_size < len(file_paths):
            file_paths = file_paths[:sample_size]
            print(f"Ограничено до: {len(file_paths)} файлов")
        
        if not file_paths:
            print("Нет файлов для анализа")
            return pd.DataFrame()
        
        if self.use_spark:
            return self._scan_with_spark(file_paths)
        else:
            return self._scan_with_threads(file_paths)
    
    def _scan_with_spark(self, file_paths: List[str]) -> pd.DataFrame:
        print(f"Анализ файлов с помощью Spark...")
        
        rdd = self.spark.sparkContext.parallelize(file_paths, numSlices=self.max_workers * 2)
        
        def process_file_spark(file_path: str) -> Dict:
            try:
                with open(file_path, 'rb') as f:
                    content = f.read()
                result = analyze_file_content(file_path, content, self.detect_faces)
                return result
            except Exception as e:
                path = Path(file_path)
                file_stat = path.stat() if path.exists() else None
                return {
                    'path': file_path,
                    'name': path.name,
                    'size': file_stat.st_size if file_stat else 0,
                    'modified_time': datetime.fromtimestamp(file_stat.st_mtime).strftime('%b %d %H:%M') if file_stat else '',
                    'success': False,
                    'ext': path.suffix.lower().lstrip('.'),
                    'text_length': 0,
                    'categories': {},
                    'total_hits': 0,
                    'uz': 'нет ПДн',
                    'face_count': 0,
                    'error': str(e)[:200]
                }
            
        results_rdd = rdd.map(process_file_spark)
        results = results_rdd.collect()
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            print("\nПример результатов:")
            print(df.head(5).to_string())
        
        return df
    
    def _scan_with_threads(self, file_paths: List[str]) -> pd.DataFrame:
        print(f"Анализ файлов (потоков: {self.max_workers})...")
        
        results = []
        lock = threading.Lock()
        processed = 0
        
        def process_file(file_path: str):
            nonlocal processed
            try:
                with open(file_path, 'rb') as f:
                    content = f.read()
                result = analyze_file_content(file_path, content, self.detect_faces)
                with lock:
                    processed += 1
                    if processed % 10 == 0:
                        print(f"  Обработано {processed}/{len(file_paths)} файлов")
                return result
            except Exception as e:
                with lock:
                    processed += 1
                path = Path(file_path)
                file_stat = path.stat() if path.exists() else None
                return {
                    'path': file_path,
                    'name': path.name,
                    'size': file_stat.st_size if file_stat else 0,
                    'modified_time': datetime.fromtimestamp(file_stat.st_mtime).strftime('%b %d %H:%M') if file_stat else '',
                    'success': False,
                    'ext': path.suffix.lower().lstrip('.'),
                    'text_length': 0,
                    'categories': {},
                    'total_hits': 0,
                    'uz': 'нет ПДн',
                    'face_count': 0,
                    'error': str(e)[:200]
                }
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(process_file, fp): fp for fp in file_paths}
            for future in as_completed(futures):
                results.append(future.result())
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            print("\nПример результатов:")
            print(df.head(5).to_string())
        
        return df
    
    def get_report_df(self, df: pd.DataFrame) -> pd.DataFrame:
        """Формирование отчета"""
        if df.empty:
            return df
        
        report_df = df.copy()
        report_df['категории_ПДн'] = report_df['categories'].apply(
            lambda x: ', '.join([f"{k}:{v}" for k, v in x.items()]) if x else ''
        )
        report_df = report_df.rename(columns={
            'path': 'путь',
            'name': 'имя_файла',
            'size': 'размер',
            'modified_time': 'время_изменения',
            'total_hits': 'количество_находок',
            'uz': 'УЗ',
            'ext': 'формат_файла',
            'face_count': 'количество_лиц'
        })
        
        columns = ['размер', 'время_изменения', 'имя_файла', 'путь', 'категории_ПДн', 
                   'количество_находок', 'количество_лиц', 'УЗ', 'формат_файла']
        return report_df[columns]
    
    def save_special_format(self, df: pd.DataFrame, output_path: str) -> str:
        if df.empty:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write("size,time,name\n")
            return output_path
        
        success_df = df[df['success'] == True]
        
        special_df = pd.DataFrame({
            'size': success_df['size'],
            'time': success_df['modified_time'],
            'name': success_df['name']
        })
        
        special_df = special_df.sort_values('size', ascending=False)
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        special_df.to_csv(output_path, index=False)
        
        print(f"Специальный отчет сохранен: {output_path}")
        return output_path
    
    def save_report_csv(self, df: pd.DataFrame, output_path: str) -> str:
        report_df = self.get_report_df(df)
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        report_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"CSV отчет сохранен: {output_path}")
        return output_path
    
    def save_report_json(self, df: pd.DataFrame, output_path: str) -> str:
        report_df = self.get_report_df(df)
        
        json_data = {
            'report_generated': datetime.now().isoformat(),
            'total_files': len(report_df),
            'files_with_pdn': len(report_df[report_df['количество_находок'] > 0]),
            'total_faces_detected': int(report_df['количество_лиц'].sum()) if 'количество_лиц' in report_df else 0,
            'results': report_df.to_dict(orient='records')
        }
        
        stats = self.get_statistics(df)
        json_data['statistics'] = stats
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        
        print(f"JSON отчет сохранен: {output_path}")
        return output_path
    
    def save_report_markdown(self, df: pd.DataFrame, output_path: str) -> str:
        report_df = self.get_report_df(df)
        stats = self.get_statistics(df)
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        md_lines = []
        
        md_lines.append("# Отчет по сканированию персональных данных\n")
        md_lines.append(f"**Дата генерации:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        md_lines.append("---\n")
        
        md_lines.append("## Общая статистика\n")
        md_lines.append("| Показатель | Значение |")
        md_lines.append("|------------|----------|")
        md_lines.append(f"| Всего файлов | {stats['total_files']} |")
        md_lines.append(f"| Успешно обработано | {stats['successful']} |")
        md_lines.append(f"| Ошибок | {stats['failed']} |")
        md_lines.append(f"| **Файлов с ПДн** | **{stats['files_with_pdn']}** |")
        md_lines.append(f"| **Всего находок ПДн** | **{stats['total_pdn_hits']}** |")
        md_lines.append(f"| **Обнаружено лиц** | **{stats['total_faces']}** |")
        md_lines.append("")
        
        md_lines.append("## Топ-10 файлов с наибольшим количеством находок\n")
        top_files = report_df.nlargest(10, 'количество_находок')
        if not top_files.empty:
            md_lines.append("| # | Файл | Размер | Время | Находок | Лиц | УЗ |")
            md_lines.append("|---|------|--------|-------|---------|-----|----|")
            for i, row in top_files.iterrows():
                size_kb = row['размер'] / 1024 if row['размер'] > 1024 else row['размер']
                size_str = f"{size_kb:.1f} KB" if row['размер'] > 1024 else f"{row['размер']} B"
                md_lines.append(f"| {i+1} | {row['имя_файла'][:30]} | {size_str} | {row['время_изменения']} | {row['количество_находок']} | {row['количество_лиц']} | {row['УЗ']} |")
        md_lines.append("")

        md_lines.append("## Список файлов в формате size,time,name\n")
        md_lines.append("```")
        md_lines.append("size,time,name")
        for _, row in report_df.sort_values('размер', ascending=False).iterrows():
            md_lines.append(f"{row['размер']},{row['время_изменения']},{row['имя_файла']}")
        md_lines.append("```")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(md_lines))
        
        print(f"Markdown отчет сохранен: {output_path}")
        return output_path
    
    def save_all_reports(self, df: pd.DataFrame, base_output_path: str) -> Dict[str, str]:
        base_path = Path(base_output_path)
        base_name = base_path.stem
        
        reports = {}
        
        csv_path = base_path.parent / f"{base_name}.csv"
        reports['csv'] = self.save_report_csv(df, str(csv_path))
        
        json_path = base_path.parent / f"{base_name}.json"
        reports['json'] = self.save_report_json(df, str(json_path))
        
        md_path = base_path.parent / f"{base_name}.md"
        reports['markdown'] = self.save_report_markdown(df, str(md_path))
        
        special_path = base_path.parent / f"{base_name}_special.csv"
        reports['special'] = self.save_special_format(df, str(special_path))
        
        return reports
    
    def get_statistics(self, df: pd.DataFrame) -> Dict[str, Any]:
        if df.empty:
            return {
                'total_files': 0,
                'successful': 0,
                'failed': 0,
                'files_with_pdn': 0,
                'protection_levels': {},
                'category_totals': {cat: 0 for cat in PDN_CATEGORIES},
                'total_pdn_hits': 0,
                'total_faces': 0
            }
        
        total = len(df)
        success = df['success'].sum()
        with_pdn = (df['total_hits'] > 0).sum()
        total_faces = df['face_count'].sum() if 'face_count' in df else 0
        
        uz_stats = df['uz'].value_counts().to_dict()
        
        cat_totals = {cat: 0 for cat in PDN_CATEGORIES}
        for cats in df['categories']:
            for cat, count in cats.items():
                if cat in cat_totals:
                    cat_totals[cat] += count
        
        return {
            'total_files': total,
            'successful': int(success),
            'failed': total - int(success),
            'files_with_pdn': int(with_pdn),
            'protection_levels': uz_stats,
            'category_totals': cat_totals,
            'total_pdn_hits': sum(cat_totals.values()),
            'total_faces': int(total_faces)
        }
    
    def __del__(self):
        if hasattr(self, 'spark') and self.use_spark:
            self.spark.stop()

# ============================================================================
# Основная функция
# ============================================================================

def main():
    parser = argparse.ArgumentParser(description="Сканер ПДн с поддержкой Spark и YOLO")
    parser.add_argument("--input", "-i", help="Директория для сканирования")
    parser.add_argument("--output", "-o", help="Базовое имя файла отчета")
    parser.add_argument("--sample", "-s", type=int, help="Ограничение количества файлов")
    parser.add_argument("--no-recursive", action="store_true", help="Отключить рекурсивный обход")
    parser.add_argument("--workers", "-w", type=int, default=4, help="Количество потоков")
    parser.add_argument("--format", "-f", choices=['csv', 'json', 'markdown', 'special', 'all'], 
                       default='all', help="Формат отчета")
    parser.add_argument("--no-spark", action="store_true", help="Отключить использование Spark")
    parser.add_argument("--no-faces", action="store_true", help="Отключить детекцию лиц (YOLO)")
    
    args = parser.parse_args()
    
    if args.input is None:
        args.input = str(Path.cwd() / "data")
    
    if args.output is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        args.output = f"pdn_report_{timestamp}"
    
    formats = ['csv', 'json', 'markdown', 'special'] if args.format == 'all' else [args.format]
    
    print(f"Запуск сканирования: {args.input}")
    print(f"Расширения: {', '.join(sorted(INCLUDE_EXTS))}")
    print(f"Spark: {'ВКЛ' if not args.no_spark and SPARK_AVAILABLE else 'ВЫКЛ'}")
    print(f"YOLO (детекция лиц): {'ВКЛ' if not args.no_faces and YOLO_AVAILABLE else 'ВЫКЛ'}")
    
    scanner = PDNScanner(
        max_workers=args.workers,
        use_spark=not args.no_spark,
        detect_faces=not args.no_faces
    )
    
    start_time = time.time()
    
    results = scanner.scan_directory(
        directory_path=args.input,
        recursive=not args.no_recursive,
        sample_size=args.sample
    )
    
    elapsed_time = time.time() - start_time
    
    reports_dir = Path(args.input) / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    base_path = reports_dir / args.output
    
    if 'csv' in formats:
        scanner.save_report_csv(results, str(base_path.with_suffix('.csv')))
    if 'json' in formats:
        scanner.save_report_json(results, str(base_path.with_suffix('.json')))
    if 'markdown' in formats:
        scanner.save_report_markdown(results, str(base_path.with_suffix('.md')))
    if 'special' in formats:
        scanner.save_special_format(results, str(base_path.with_suffix('_special.csv')))
    
    stats = scanner.get_statistics(results)
    
    print(f"Всего файлов: {stats['total_files']}")
    print(f"Успешно обработано: {stats['successful']}")
    print(f"Файлов с ПДн: {stats['files_with_pdn']}")
    print(f"Всего находок: {stats['total_pdn_hits']}")
    print(f"Обнаружено лиц: {stats['total_faces']}")
    print(f"Время выполнения: {elapsed_time:.2f} сек")
    
    print(f"\nОтчеты сохранены в: {reports_dir}")

if __name__ == "__main__":
    main()